Using: Correlates of War Project, National Material Capabilities (NMC) Data Version 6.0 (Period Covered: 1816-2016)

In [62]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_breusch_godfrey

# Load and clean data

df = pd.read_csv(
    "/Users/kylejonespatricia/Downloads/NMC_Documentation-6.0/NMC-60-abridged/NMC-60-abridged.csv"
)
df = df[["milex", "irst", "pec"]].dropna()
# Define dependent and independent variables
Y = df["milex"]
X = df[["irst", "pec"]]
X = sm.add_constant(X)
# Fit OLS model
ols_model = sm.OLS(Y, X).fit()
print(ols_model.summary())
lm_test = acorr_breusch_godfrey(ols_model, nlags=3)
lm_stat, lm_pvalue, f_stat, f_pvalue = lm_test
print(f"LM Statistic: {lm_stat:.4f}, p-value: {lm_pvalue:.4f}")
nw_model = ols_model.get_robustcov_results(cov_type="HAC", maxlags=3)
print(nw_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  milex   R-squared:                       0.723
Model:                            OLS   Adj. R-squared:                  0.723
Method:                 Least Squares   F-statistic:                 2.085e+04
Date:                Sat, 08 Mar 2025   Prob (F-statistic):               0.00
Time:                        20:56:41   Log-Likelihood:            -2.8242e+05
No. Observations:               15951   AIC:                         5.648e+05
Df Residuals:                   15948   BIC:                         5.649e+05
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -9.649e+05   9.57e+04    -10.079      0.0

In [15]:
import io

from matplotlib import pyplot as plt

# Capture the model summary output

buffer = io.StringIO()

print(nw_model.summary(), file=buffer)

summary_text = buffer.getvalue()


# Render the text as an image using Matplotlib

plt.figure(figsize=(10, 7))

plt.axis("off")  # Hide axes

plt.text(0, 0, summary_text, fontsize=10, family="monospace")

plt.savefig("model_summary.png")

In [25]:
# Apply differencing
df["milex_diff"] = df["milex"].diff()
df = df.dropna()
# Fit OLS on differenced data
Y_diff = df["milex_diff"]
X_diff = df[["irst", "pec"]]
X_diff = sm.add_constant(X_diff)
ols_diff_model = sm.OLS(Y_diff, X_diff).fit()

buffer = io.StringIO()
print(ols_diff_model.summary(), file=buffer)

summary_text = buffer.getvalue()


# Render the text as an image using Matplotlib

plt.figure(figsize=(10, 7))

plt.axis("off")  # Hide axes

plt.text(0, 0, summary_text, fontsize=10, family="monospace")

plt.savefig("model_summary.png")

In [27]:
lm_test_diff = acorr_breusch_godfrey(ols_diff_model, nlags=3)
print(
    f"LM Statistic (Differenced): {lm_test_diff[0]:.4f}, p-value: {lm_test_diff[1]:.4f}"
)

LM Statistic (Differenced): 91.2065, p-value: 0.0000


In [29]:
nw_diff_model = ols_diff_model.get_robustcov_results(cov_type="HAC", maxlags=3)
print(nw_diff_model.summary())

                            OLS Regression Results                            
Dep. Variable:             milex_diff   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     5.111
Date:                Sat, 08 Mar 2025   Prob (F-statistic):            0.00604
Time:                        20:34:24   Log-Likelihood:            -2.7071e+05
No. Observations:               15946   AIC:                         5.414e+05
Df Residuals:                   15943   BIC:                         5.414e+05
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1.236e+05   5.16e+04     -2.397      0.0

In [31]:
buffer = io.StringIO()
print(nw_diff_model.summary(), file=buffer)

summary_text = buffer.getvalue()


# Render the text as an image using Matplotlib

plt.figure(figsize=(10, 7))

plt.axis("off")  # Hide axes

plt.text(0, 0, summary_text, fontsize=10, family="monospace")

plt.savefig("model_summary.png")

In [48]:
import statsmodels.graphics.tsaplots as tsaplots

# Filter dataset for only the USA
df = pd.read_csv(
    "/Users/kylejonespatricia/Downloads/NMC_Documentation-6.0/NMC-60-abridged/NMC-60-abridged.csv"
)
df_usa = df[df["stateabb"] == "USA"].copy()

# Apply differencing
df_usa["milex_diff"] = df_usa["milex"].diff()
df_usa = df_usa.dropna()

# Define dependent and independent variables for USA model
Y_usa = df_usa["milex"]
X_usa = df_usa[["irst", "pec"]]
X_usa = sm.add_constant(X_usa)

# Fit OLS for USA data
ols_usa_model = sm.OLS(Y_usa, X_usa).fit()

# Fit OLS for differenced USA data
Y_usa_diff = df_usa["milex_diff"]
X_usa_diff = df_usa[["irst", "pec"]]
X_usa_diff = sm.add_constant(X_usa_diff)

ols_usa_diff_model = sm.OLS(Y_usa_diff, X_usa_diff).fit()

# Extract residuals
residuals_ols_usa = ols_usa_model.resid
residuals_ols_usa_diff = ols_usa_diff_model.resid

# Create plots for the USA
fig, ax = plt.subplots(figsize=(10, 5))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_position(("outward", 5))
ax.spines["bottom"].set_position(("outward", 5))
ax.plot(
    df_usa["year"],
    df_usa["milex"],
    markersize=3,
    color="black",
    label="Military Expenditures (USA)",
)
ax.set_title("Military Expenditures Over Time for the USA")
plt.savefig("milex_usa_over_time.png")
plt.show()

# ACF Plot for OLS Residuals (USA)
fig, ax = plt.subplots(figsize=(10, 5))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_position(("outward", 5))
ax.spines["bottom"].set_position(("outward", 5))
tsaplots.plot_acf(residuals_ols_usa, lags=20, alpha=0.05, ax=ax)
ax.set_title("Autocorrelation of OLS Residuals (USA)")
ax.set_xlabel("Lag")
ax.set_ylabel("Autocorrelation")
plt.savefig("ols_residuals_usa_acf.png")
plt.show()

# ACF Plot for Differenced Residuals (USA)
fig, ax = plt.subplots(figsize=(10, 5))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_position(("outward", 5))
ax.spines["bottom"].set_position(("outward", 5))
tsaplots.plot_acf(residuals_ols_usa_diff, lags=20, alpha=0.05, ax=ax)
ax.set_title("Autocorrelation of Differenced Residuals (USA)")
ax.set_xlabel("Lag")
ax.set_ylabel("Autocorrelation")
plt.savefig("differenced_residuals_usa_acf.png")
plt.show()

# Compare Military Expenditures Before and After Differencing (USA)
fig, ax = plt.subplots(figsize=(10, 5))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_position(("outward", 5))
ax.spines["bottom"].set_position(("outward", 5))
ax.plot(
    df_usa["year"],
    df_usa["milex"],
    label="Original MILEX (USA)",
    color="black",
    linestyle="-",
)
ax.plot(
    df_usa["year"],
    df_usa["milex_diff"],
    label="Differenced MILEX (USA)",
    color="gray",
    linestyle="--",
)
ax.set_title("Military Expenditures: Original vs Differenced (USA)")
plt.savefig("milex_usa_original_vs_diff.png")
plt.show()

In [37]:
df.head()

,index,milex,irst,pec,milex_diff
0,5,1612,100,321,56.0
1,6,1079,100,332,-533.0
2,7,1170,110,345,91.0
3,8,1261,110,390,91.0
4,9,1336,120,424,75.0


In [50]:
df.head()

,stateabb,ccode,year,milex,milper,irst,pec,tpop,upop,cinc,version
0,USA,2,1816,3823,17,80,254,8659.0,101.0,0.039697,2021
1,USA,2,1817,2466,15,80,277,8899.0,106.0,0.035817,2021
2,USA,2,1818,1910,14,90,302,9139.0,112.0,0.036127,2021
3,USA,2,1819,2301,13,90,293,9379.0,118.0,0.037133,2021
4,USA,2,1820,1556,15,110,303,9618.0,124.0,0.037087,2021


In [58]:
df["stateabb"].nunique()

217